# Data Ingestion

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/Project/training/best_model/model_state_dict.pth" "/content/model_state_dict.pth"

In [ ]:
!cp "/content/drive/MyDrive/Project/data_clean/data_clean_v2.zip" "/content/data_clean.zip"

!unzip -q "/content/data_clean.zip" -d "/content/data_clean/"

# Exploratory Data Analysis

Library Imports

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.transforms import v2 as T
import torchvision.transforms.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from sklearn.calibration import calibration_curve
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

Evaluate the model through the model_state_dict.pth file

In [ ]:
reports_dir = '/content/reports'
os.makedirs(reports_dir, exist_ok=True)
print(f"Reports directory ensured at: {reports_dir}")

class ResizeAndPad:
    def __init__(self, target_size, fill=0):
        self.target_h, self.target_w = target_size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        if w > self.target_w:
            img.thumbnail((self.target_w, h))
            w, h = img.size
        if h > self.target_h:
            img.thumbnail((w, self.target_h))
            w, h = img.size
        pad_h = max(self.target_h - h, 0)
        pad_w = max(self.target_w - w, 0)
        padding = (pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2)
        return F.pad(img, padding, fill=self.fill)

img_size = 150
base_transforms = T.Compose([
    ResizeAndPad((img_size, img_size)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class DeepFakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

model = DeepFakeDetector()
model.load_state_dict(torch.load('/content/model_state_dict.pth', map_location=device))
model = model.to(device)
model.eval()

test_dir = '/content/data_clean/data_clean/Test'

test_dataset = ImageFolder(root=test_dir, transform=base_transforms)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Class mapping: {test_dataset.class_to_idx}")
print(f"Total test images: {len(test_dataset)}")

all_labels = []
all_probs = []

print("Running evaluation on Test Set...")
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        labels = labels.to(device)


        outputs = model(images)
        probs = torch.sigmoid(outputs).squeeze()

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


all_preds = [1 if p > 0.5 else 0 for p in all_probs]

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\n✅ True Test Accuracy: {test_acc * 100:.2f}%")

Probability Distribution Histogram




In [ ]:
plt.figure(figsize=(8,4))
plt.hist([p for p,l in zip(all_probs, all_labels) if l==0], bins=30, alpha=0.6, label="Real")
plt.hist([p for p,l in zip(all_probs, all_labels) if l==1], bins=30, alpha=0.6, label="Fake")
plt.title("Probability Distribution by Class")
plt.xlabel("Predicted Probability (Fake)")
plt.ylabel("Count")
plt.legend()
pdh_path = os.path.join(reports_dir, 'probability_distribution.png')
plt.savefig(pdh_path, bbox_inches='tight')
plt.show()

Misclassified Image Gallery

In [ ]:
misclassified = [i for i,(p,l) in enumerate(zip(all_preds, all_labels)) if p != l]

images = [test_dataset[i][0] for i in misclassified[:16]]
grid = make_grid(images, nrow=4)

plt.figure(figsize=(8,8))
plt.imshow(grid.permute(1,2,0))
plt.title("Misclassified Images")
plt.axis("off")
missed_path = os.path.join(reports_dir, 'misclassified_16.png')
plt.savefig(missed_path, bbox_inches='tight')
plt.show()

UMAP Embeddding Plot

In [ ]:
import umap

In [ ]:
embeddings = []
labels = []

with torch.no_grad():
    for images, lbls in test_loader:
        images = images.to(device)
        feats = model.features(images).squeeze()
        embeddings.append(feats.cpu())
        labels.extend(lbls.numpy())

embeddings = torch.cat(embeddings).numpy()

reducer = umap.UMAP()
emb_2d = reducer.fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(emb_2d[:,0], emb_2d[:,1], c=labels, cmap="coolwarm", alpha=0.6)
plt.title("UMAP Embedding: Real vs Fake")
plt.colorbar(label="Class (0=Real, 1=Fake)")
umap_path = os.path.join(reports_dir, 'UMAP_embedding.png')
plt.savefig(umap_path, bbox_inches='tight')
plt.show()

Threshold Sweep Plot

In [ ]:
thresholds = np.linspace(0,1,100)
accs, precs, recs = [], [], []

for t in thresholds:
    preds = (np.array(all_probs) > t).astype(int)
    accs.append(accuracy_score(all_labels, preds))
    precs.append(precision_score(all_labels, preds))
    recs.append(recall_score(all_labels, preds))

plt.figure(figsize=(10,6))
plt.plot(thresholds, accs, label="Accuracy")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Sweep")
plt.legend()
thresh_path = os.path.join(reports_dir, 'threshold_sweep.png')
plt.savefig(thresh_path, bbox_inches='tight')
plt.show()

Save Images in reports to MyDrive

In [ ]:
!cp "/content/reports/probability_distribution.png" "/content/drive/MyDrive/Project/reports/"

In [ ]:
!cp "/content/reports/misclassified_16.png" "/content/drive/MyDrive/Project/reports/"

In [ ]:
!cp "/content/reports/UMAP_embedding.png" "/content/drive/MyDrive/Project/reports/"

In [ ]:
!cp "/content/reports/threshold_sweep.png" "/content/drive/MyDrive/Project/reports/"